# vzviz: MUR SST Dashboard

A simple example showing how to visualize a remote MUR SST NetCDF file with `vzviz`.

Run with [juv](https://github.com/manzt/juv):
```bash
juv run examples/mur_sst_dashboard.ipynb
```

In [ ]:
# /// script
# requires-python = ">=3.11"
# dependencies = [
#     "earthaccess",
#     "virtualizarr[hdf] @ /Users/max/Documents/Code/claude-workspaces/virtualizarr/VirtualiZarr",
#     "vzviz @ /Users/max/Documents/Code/claude-workspaces/virtualizarr/vzviz",
#     "obspec-utils @ git+https://github.com/virtual-zarr/obspec-utils@store-loop",
#     "aiohttp",
#     "pandas",
#     "jupyterlab",
# ]
# ///

## Setup

Authenticate with NASA Earthdata and create a ManifestStore from a remote MUR SST file.

In [ ]:
from urllib.parse import urlparse

import earthaccess
import virtualizarr as vz
import vzviz
import holoviews as hv
import panel as pn

from obspec_utils.registry import ObjectStoreRegistry
from obspec_utils.stores import AiohttpStore

hv.extension("bokeh")
pn.extension()

In [ ]:
# Authenticate with NASA Earthdata
earthaccess.login()

In [ ]:
# Search for MUR SST data
results = earthaccess.search_data(
    concept_id="C1996881146-POCLOUD", count=1, temporal=("2002-06-01", "2002-06-01")
)

# Get the HTTPS URL
https_links = earthaccess.results.DataGranule.data_links(results[0], access="external")
https_url = https_links[0]
print(f"URL: {https_url}")

In [ ]:
# Parse URL and get auth token
parsed = urlparse(https_url)
base_url = f"{parsed.scheme}://{parsed.netloc}"
token = earthaccess.get_edl_token()["access_token"]

# Create store with authentication
store = AiohttpStore(
    base_url,
    headers={"Authorization": f"Bearer {token}"},
)
registry = ObjectStoreRegistry({base_url: store})

In [ ]:
# Create ManifestStore by parsing the NetCDF file
parser = vz.parsers.HDFParser()
manifest_store = parser(https_url, registry=registry)
print("ManifestStore created!")

## Dashboard

Launch the interactive dashboard to explore the chunk manifest.

In [ ]:
dashboard = vzviz.manifest_dashboard(manifest_store, variable="analysed_sst")
dashboard

### Serve in Browser

To serve the dashboard in a separate browser window:

In [ ]:
# Uncomment to serve in browser:
# dashboard.show()